In [ ]:
# Install Captum if not already installed
# Uncomment the following line if running in an environment where Captum is not installed
!pip install captum

# Import necessary libraries
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image
from captum.attr import DeepLift
import cv2

# Check for GPU availability and set device accordingly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Function to preprocess images
def preprocess_image(image_path):
    """
    Preprocesses the input image for VGG16.

    Args:
        image_path (str): Path to the input image.

    Returns:
        torch.Tensor: Preprocessed image tensor.
    """
    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # Resize to VGG16 input size
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                             std=[0.229, 0.224, 0.225]),
    ])
    image = Image.open(image_path).convert('RGB')  # Ensure image is in RGB
    image_tensor = transform(image).unsqueeze(0)  # Add batch dimension
    return image_tensor.to(device)

# Define the modified VGG16 model for binary classification
class VGG16Modified(nn.Module):
    def __init__(self):
        super(VGG16Modified, self).__init__()
        # Load pre-trained VGG16 model
        self.vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        # Modify the classifier to output 2 classes
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_ftrs, 2)  # Binary classification
        )

    def forward(self, x):
        return self.vgg(x)

# Function to load the trained model
def load_model(checkpoint_path):
    """
    Loads the trained VGG16Modified model from the checkpoint.

    Args:
        checkpoint_path (str): Path to the model checkpoint.

    Returns:
        nn.Module: Loaded model in evaluation mode.
    """
    model = VGG16Modified()
    # Load state dictionary with appropriate device mapping
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    return model

# Define class names
class_names = ['Glaucous_Winged_Gull', 'Slaty_Backed_Gull']

# DeepLIFT function using Captum's DeepLift
def deep_lift(model, image_tensor, target_class):
    """
    Computes the DeepLIFT attribution for a given image and target class.

    Args:
        model (nn.Module): The trained model.
        image_tensor (torch.Tensor): Preprocessed image tensor.
        target_class (int): Index of the target class for attribution.

    Returns:
        np.ndarray: Attribution map as a NumPy array.
    """
    # Initialize DeepLift
    deep_lift = DeepLift(model)
    # Define baseline (zero tensor)
    baseline = torch.zeros_like(image_tensor).to(device)
    # Compute attributions
    attr, delta = deep_lift.attribute(
        image_tensor,
        baselines=baseline,
        target=target_class,
        return_convergence_delta=True
    )
    return attr.squeeze().cpu().detach().numpy()

# Function to process and separate positive and negative attributions
def process_deep_lift_map(deep_lift_map):
    """
    Processes the raw DeepLIFT attribution map by separating positive and negative attributions.

    Args:
        deep_lift_map (np.ndarray): Raw attribution map.

    Returns:
        tuple: (positive_attribution, negative_attribution) as NumPy arrays.
    """
    # Separate positive and negative attributions
    pos_attr = np.sum(np.maximum(deep_lift_map, 0), axis=0)
    neg_attr = np.sum(np.minimum(deep_lift_map, 0), axis=0)

    # Normalize positive attributions to [0, 1]
    pos_attr = (pos_attr - pos_attr.min()) / (pos_attr.max() - pos_attr.min() + 1e-8)

    # Normalize negative attributions to [0, 1]
    neg_attr = (neg_attr - neg_attr.min()) / (neg_attr.max() - neg_attr.min() + 1e-8)

    return pos_attr, neg_attr

# Function to plot and visualize the results
def plot_comparative_visualizations(image_path, pos_attr, neg_attr, predicted_class):
    """
    Plots the original image, DeepLIFT positive attribution heatmap, negative attribution heatmap, and overlays.

    Args:
        image_path (str): Path to the input image.
        pos_attr (np.ndarray): Positive attribution map.
        neg_attr (np.ndarray): Negative attribution map.
        predicted_class (int): Index of the predicted class.
    """
    # Open and resize the original image
    image = Image.open(image_path).convert('RGB').resize((224, 224))
    image = np.array(image)

    # Create heatmaps for positive and negative attributions
    heatmap_pos = cv2.applyColorMap(np.uint8(255 * pos_attr), cv2.COLORMAP_JET)
    heatmap_pos = cv2.cvtColor(heatmap_pos, cv2.COLOR_BGR2RGB)
    overlay_pos = (0.6 * image + 0.4 * heatmap_pos).astype(np.uint8)

    heatmap_neg = cv2.applyColorMap(np.uint8(255 * neg_attr), cv2.COLORMAP_JET)
    heatmap_neg = cv2.cvtColor(heatmap_neg, cv2.COLOR_BGR2RGB)
    overlay_neg = (0.6 * image + 0.4 * heatmap_neg).astype(np.uint8)

    # Combined Attribution Overlay
    combined_heatmap = heatmap_pos * pos_attr[..., np.newaxis] + heatmap_neg * (-neg_attr[..., np.newaxis])
    combined_heatmap = combined_heatmap / (np.max(combined_heatmap) + 1e-8)  # Normalize to [0,1]
    overlay_combined = (0.6 * image + 0.4 * combined_heatmap).astype(np.uint8)

    # Plotting
    plt.figure(figsize=(24, 6))

    # Original Image with Predicted Class
    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title(f"Original Image\nPredicted Class: {class_names[predicted_class]}")
    plt.axis('off')

    # Positive Attribution Heatmap
    plt.subplot(1, 4, 2)
    plt.imshow(pos_attr, cmap='hot')
    plt.title("Positive DeepLIFT Attribution")
    plt.axis('off')
    plt.colorbar(fraction=0.046, pad=0.04)

    # Negative Attribution Heatmap
    plt.subplot(1, 4, 3)
    plt.imshow(neg_attr, cmap='cool')
    plt.title("Negative DeepLIFT Attribution")
    plt.axis('off')
    plt.colorbar(fraction=0.046, pad=0.04)

    # Overlay of Positive and Negative Heatmaps on Original Image
    plt.subplot(1, 4, 4)
    plt.imshow(overlay_combined)
    plt.title("Combined DeepLIFT Attribution Overlay")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Function to process all images in the test folder
def process_test_folder(test_folder_path, model):
    """
    Processes each image in the test folder, generates DeepLIFT attributions, and visualizes them.

    Args:
        test_folder_path (str): Path to the folder containing test images.
        model (nn.Module): The trained model.
    """
    # Iterate through all files in the test folder
    for root, _, files in os.walk(test_folder_path):
        for file in files:
            # Check for image file extensions
            if not file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                print(f"Skipping non-image file: {file}")
                continue

            image_path = os.path.join(root, file)
            print(f"\nProcessing: {image_path}")

            try:
                # Preprocess the image
                image_tensor = preprocess_image(image_path)

                # Get model prediction
                with torch.no_grad():
                    output = model(image_tensor)
                    probabilities = torch.softmax(output, dim=1)
                    predicted_class_idx = torch.argmax(probabilities, dim=1).item()
                    predicted_class_prob = probabilities[0, predicted_class_idx].item()
                    print(f"Predicted Class: {class_names[predicted_class_idx]} (Probability: {predicted_class_prob:.4f})")

                # Generate DeepLIFT attribution map for the predicted class
                deep_lift_map = deep_lift(model, image_tensor, predicted_class_idx)

                # Process attribution map to get positive and negative contributions
                pos_attr, neg_attr = process_deep_lift_map(deep_lift_map)

                # Visualize the results
                plot_comparative_visualizations(image_path, pos_attr, neg_attr, predicted_class_idx)

            except Exception as e:
                print(f"Error processing {image_path}: {e}")

# Main execution block
if __name__ == "__main__":
    # Path to the trained model checkpoint
    checkpoint_path = '/content/drive/My Drive/FYP/VGGModel/HQ2ltst_20241123/best_model_vgg_20241123.pth'

    # Load the model
    model = load_model(checkpoint_path)
    print("Model loaded successfully.")

    # Path to the test images folder
    test_folder_path = '/content/drive/My Drive/FYP/Dataset/HQ2/test/Slaty_Backed_Gull'

    # Process and visualize DeepLIFT attributions for all images in the test folder
    process_test_folder(test_folder_path, model)


Output hidden; open in https://colab.research.google.com to view.